# Notebook 67 — Policy Learning

**Confidence-Scheduled Decoding**  
**Position:** Notebook 61 Telemetry Dataset → **Notebook 67 Policy Learning** → Notebook 71 Offline Evaluation

## Engineering statement

**Telemetry specifies admissible policy.**

Notebook 61 produced telemetry rows. Notebook 67 learns where decoding actions are admissible. Notebook 71 tests the learned policy under offline replay.

The policy is a decision boundary over decoding state:

\[
s_t = (\text{confidence}, \text{entropy}, \text{margin}, \text{risk}, \text{budget}, \text{cost}, \text{verifier disagreement})
\]

with actions:

\[
a_t \in \{\text{continue}, \text{deepen}, \text{verify}, \text{stop}, \text{fallback}\}.
\]

Notebook 67 stops after producing a learned policy, diagnostics, figures, and a replay handoff table.

## 1. Setup

All figures are written to `figures/`.

This notebook is self-contained. In the repo workflow, replace the synthetic telemetry cell with the Notebook 61 telemetry export.

In [ ]:
from pathlib import Path
import json
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import LabelEncoder

ROOT = Path.cwd()
FIG_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"
ARTIFACT_DIR = ROOT / "artifacts"

for d in [FIG_DIR, DATA_DIR, ARTIFACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(67)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print("Notebook 67 ready")

## 2. Telemetry schema

Notebook 61 emits decision-point rows. Notebook 67 consumes them.

The policy observes only fields available at the current decoding step. Future reward is exported for Notebook 71 replay, not used as a policy feature.

In [ ]:
telemetry_schema = pd.DataFrame([
    ("request_id", "logical request identifier"),
    ("step", "decode step / policy decision index"),
    ("domain", "task family"),
    ("confidence", "model confidence proxy; larger is safer"),
    ("entropy", "next-token uncertainty proxy; larger is less certain"),
    ("margin", "top-1 minus top-2 probability margin"),
    ("risk_score", "external/user/system risk score"),
    ("latency_budget_ms", "remaining request latency budget"),
    ("tokens_so_far", "generated tokens before this decision"),
    ("cost_so_far", "normalized accumulated compute cost"),
    ("verifier_disagreement", "checker/model disagreement signal"),
    ("chosen_action", "observed or hindsight policy label"),
    ("reward", "offline scalar utility used later by Notebook 71"),
], columns=["field", "meaning"])

telemetry_schema

## 3. Synthetic Notebook-61-like telemetry

Qualitative rules:

- high confidence, low risk → **continue**
- uncertainty with budget → **deepen**
- risk or verifier disagreement → **verify**
- high confidence with budget pressure → **stop**
- low confidence and low budget → **fallback**

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

domains = np.array(["math", "code", "qa", "safety", "long_context"])
n = 3600

domain = RNG.choice(domains, size=n, p=[0.24, 0.20, 0.26, 0.14, 0.16])

domain_risk = {"math": 0.24, "code": 0.40, "qa": 0.31, "safety": 0.78, "long_context": 0.52}
domain_entropy_shift = {"math": -0.06, "code": 0.09, "qa": 0.00, "safety": 0.18, "long_context": 0.23}

step = RNG.integers(1, 96, size=n)
tokens_so_far = step + RNG.poisson(18, size=n)
latency_budget_ms = np.clip(RNG.normal(900 - 5.6 * step, 190, size=n), 80, 1400)

base_confidence = sigmoid(RNG.normal(0.75, 1.0, size=n) - 0.012 * step)
confidence = np.clip(base_confidence - np.vectorize(domain_entropy_shift.get)(domain) * 0.20, 0.02, 0.995)

entropy = np.clip(
    1.55 - 1.25 * confidence + np.vectorize(domain_entropy_shift.get)(domain) + RNG.normal(0, 0.16, size=n),
    0.05,
    2.2,
)

margin = np.clip(0.70 * confidence - 0.22 * entropy + RNG.normal(0.06, 0.12, size=n), 0.00, 0.95)

risk_score = np.clip(
    np.vectorize(domain_risk.get)(domain) + RNG.normal(0, 0.10, size=n) + 0.18 * (entropy > 1.10),
    0,
    1,
)

cost_so_far = np.clip(0.0017 * tokens_so_far + 0.0009 * step + RNG.normal(0.02, 0.015, size=n), 0, 1)

verifier_disagreement = np.clip(
    0.18 + 0.45 * (entropy > 1.00) + 0.22 * (risk_score > 0.65) + RNG.normal(0, 0.13, size=n),
    0,
    1,
)

budget_pressure = 1 - (
    (latency_budget_ms - latency_budget_ms.min())
    / (latency_budget_ms.max() - latency_budget_ms.min())
)

actions = []
for c, e, m, r, b, v in zip(confidence, entropy, margin, risk_score, budget_pressure, verifier_disagreement):
    if r > 0.72 or v > 0.72:
        action = "verify"
    elif b > 0.82 and c > 0.62 and m > 0.24:
        action = "stop"
    elif c < 0.42 and b > 0.35:
        action = "deepen"
    elif c < 0.38 and b <= 0.35:
        action = "fallback"
    elif e > 1.10 and m < 0.22:
        action = "deepen"
    else:
        action = "continue"

    if RNG.random() < 0.08:
        action = RNG.choice(["continue", "deepen", "verify", "stop", "fallback"], p=[0.38, 0.24, 0.18, 0.14, 0.06])

    actions.append(action)

utility_by_action = {"continue": 0.74, "deepen": 0.80, "verify": 0.77, "stop": 0.70, "fallback": 0.50}

reward = np.array([utility_by_action[a] for a in actions])
reward += 0.18 * confidence + 0.10 * margin
reward -= 0.16 * entropy + 0.11 * cost_so_far + 0.08 * budget_pressure
reward += RNG.normal(0, 0.045, size=n)
reward = np.clip(reward, 0, 1)

telemetry = pd.DataFrame({
    "request_id": [f"req_{i//8:05d}" for i in range(n)],
    "step": step,
    "domain": domain,
    "confidence": confidence,
    "entropy": entropy,
    "margin": margin,
    "risk_score": risk_score,
    "latency_budget_ms": latency_budget_ms,
    "tokens_so_far": tokens_so_far,
    "cost_so_far": cost_so_far,
    "verifier_disagreement": verifier_disagreement,
    "chosen_action": actions,
    "reward": reward,
})

telemetry.to_csv(DATA_DIR / "notebook67_policy_learning_telemetry.csv", index=False)
telemetry.head()

## 4. Dataset checks

The first check is not model accuracy. The first check is whether the telemetry covers enough policy states to learn from.

In [ ]:
summary = telemetry.describe(include="all").transpose()
summary.to_csv(DATA_DIR / "notebook67_telemetry_summary.csv")
summary

In [ ]:
action_order = ["continue", "deepen", "verify", "stop", "fallback"]

fig, ax = plt.subplots()
telemetry["chosen_action"].value_counts().reindex(action_order).plot(kind="bar", ax=ax)
ax.set_title("Notebook 67 — policy action distribution")
ax.set_xlabel("Action")
ax.set_ylabel("Telemetry rows")
fig.tight_layout()
fig.savefig(FIG_DIR / "67_00_action_distribution.png", dpi=180)
plt.show()

## 5. Figure 1 — Telemetry state space

Confidence alone does not specify the policy. The useful object is the geometry of confidence, entropy, margin, risk, budget, and verifier disagreement.

In [ ]:
fig, ax = plt.subplots()

for action, group in telemetry.groupby("chosen_action"):
    ax.scatter(
        group["entropy"],
        group["confidence"],
        s=10 + 30 * group["risk_score"],
        alpha=0.30,
        label=action,
    )

ax.set_title("67.01 — Telemetry state space")
ax.set_xlabel("Entropy")
ax.set_ylabel("Confidence")
ax.legend(markerscale=1.5, frameon=True)
fig.tight_layout()
fig.savefig(FIG_DIR / "67_01_telemetry_state_space.png", dpi=180)
plt.show()

## 6. Admissible action score

Notebook 67 frames policy learning as admissibility.

\[
A(a \mid s) =
U(a, s)
- \lambda_{cost} C(a, s)
- \lambda_{risk} R(a, s)
- \lambda_{latency} L(a, s)
\]

The learned policy approximates:

\[
\hat{\pi}(s) = \arg\max_a A(a \mid s)
\]

Notebook 67 proposes \(\hat{\pi}\). Notebook 71 tests it under replay.

In [ ]:
features = [
    "step",
    "confidence",
    "entropy",
    "margin",
    "risk_score",
    "latency_budget_ms",
    "tokens_so_far",
    "cost_so_far",
    "verifier_disagreement",
]

X_numeric = telemetry[features].copy()
X_domain = pd.get_dummies(telemetry["domain"], prefix="domain")
X = pd.concat([X_numeric, X_domain], axis=1)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(telemetry["chosen_action"])

X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X,
    y,
    telemetry.index,
    test_size=0.25,
    random_state=67,
    stratify=y,
)

policy = RandomForestClassifier(
    n_estimators=180,
    max_depth=9,
    min_samples_leaf=10,
    class_weight="balanced_subsample",
    random_state=67,
    n_jobs=-1,
)

policy.fit(X_train, y_train)

pred = policy.predict(X_test)
proba = policy.predict_proba(X_test)

metrics = {
    "accuracy": float(accuracy_score(y_test, pred)),
    "macro_f1": float(f1_score(y_test, pred, average="macro")),
    "log_loss": float(log_loss(y_test, proba)),
    "classes": label_encoder.classes_.tolist(),
}

metrics

## 7. Figure 2 — Admissible action regions

This surface visualizes the learned action boundary over confidence × entropy while holding the other fields at typical values.

In [ ]:
grid_n = 110
conf_grid = np.linspace(0.04, 0.98, grid_n)
ent_grid = np.linspace(0.05, 2.05, grid_n)

template = X.median(numeric_only=True).to_dict()
for col in X.columns:
    if col.startswith("domain_"):
        template[col] = 0
template["domain_qa"] = 1

rows = []
for e in ent_grid:
    for c in conf_grid:
        row = template.copy()
        row["confidence"] = c
        row["entropy"] = e
        row["margin"] = np.clip(0.70 * c - 0.22 * e + 0.06, 0, 0.95)
        rows.append(row)

grid = pd.DataFrame(rows)[X.columns]
grid_pred = policy.predict(grid)
grid_actions = label_encoder.inverse_transform(grid_pred).reshape(grid_n, grid_n)

action_to_int = {a: i for i, a in enumerate(label_encoder.classes_)}
Z = np.vectorize(action_to_int.get)(grid_actions)

fig, ax = plt.subplots()
im = ax.imshow(
    Z,
    origin="lower",
    aspect="auto",
    extent=[conf_grid.min(), conf_grid.max(), ent_grid.min(), ent_grid.max()],
)

ax.set_title("67.02 — Admissible action regions")
ax.set_xlabel("Confidence")
ax.set_ylabel("Entropy")

cbar = fig.colorbar(im, ax=ax, ticks=list(action_to_int.values()))
cbar.ax.set_yticklabels(list(action_to_int.keys()))

fig.tight_layout()
fig.savefig(FIG_DIR / "67_02_admissible_action_regions.png", dpi=180)
plt.show()

## 8. Figure 3 — Budget deformation

Latency pressure changes what is admissible. The same confidence/entropy state can route differently when budget is abundant versus exhausted.

In [ ]:
low_budget = float(np.percentile(telemetry["latency_budget_ms"], 12))
high_budget = float(np.percentile(telemetry["latency_budget_ms"], 88))

rows_low = []
rows_high = []

for e in ent_grid:
    for c in conf_grid:
        row_low = template.copy()
        row_low["confidence"] = c
        row_low["entropy"] = e
        row_low["margin"] = np.clip(0.70 * c - 0.22 * e + 0.06, 0, 0.95)
        row_low["latency_budget_ms"] = low_budget
        rows_low.append(row_low)

        row_high = row_low.copy()
        row_high["latency_budget_ms"] = high_budget
        rows_high.append(row_high)

Z_low = np.vectorize(action_to_int.get)(
    label_encoder.inverse_transform(policy.predict(pd.DataFrame(rows_low)[X.columns])).reshape(grid_n, grid_n)
)

Z_high = np.vectorize(action_to_int.get)(
    label_encoder.inverse_transform(policy.predict(pd.DataFrame(rows_high)[X.columns])).reshape(grid_n, grid_n)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

for ax, Z_budget, subtitle in [
    (axes[0], Z_high, f"high budget: {high_budget:.0f} ms"),
    (axes[1], Z_low, f"low budget: {low_budget:.0f} ms"),
]:
    im = ax.imshow(
        Z_budget,
        origin="lower",
        aspect="auto",
        extent=[conf_grid.min(), conf_grid.max(), ent_grid.min(), ent_grid.max()],
    )
    ax.set_title(subtitle)
    ax.set_xlabel("Confidence")

axes[0].set_ylabel("Entropy")
fig.suptitle("67.03 — Budget deformation of the policy surface")

cbar = fig.colorbar(im, ax=axes.ravel().tolist(), ticks=list(action_to_int.values()), fraction=0.046, pad=0.04)
cbar.ax.set_yticklabels(list(action_to_int.keys()))

fig.savefig(FIG_DIR / "67_03_budget_deformation.png", dpi=180, bbox_inches="tight")
plt.show()

## 9. Figure 4 — Verifier gate

Risk and verifier disagreement should not behave like ordinary continuous features. They act like gates.

As either risk or disagreement rises, **verify** becomes admissible even when confidence is not low.

In [ ]:
risk_grid = np.linspace(0, 1, 120)
disagree_grid = np.linspace(0, 1, 120)

gate_template = template.copy()
gate_template["confidence"] = float(np.percentile(telemetry["confidence"], 60))
gate_template["entropy"] = float(np.percentile(telemetry["entropy"], 45))
gate_template["margin"] = float(np.percentile(telemetry["margin"], 55))
gate_template["latency_budget_ms"] = float(np.percentile(telemetry["latency_budget_ms"], 55))

gate_rows = []
for r in risk_grid:
    for v in disagree_grid:
        row = gate_template.copy()
        row["risk_score"] = r
        row["verifier_disagreement"] = v
        gate_rows.append(row)

gate_df = pd.DataFrame(gate_rows)[X.columns]
gate_actions = label_encoder.inverse_transform(policy.predict(gate_df)).reshape(len(risk_grid), len(disagree_grid))
gate_Z = np.vectorize(action_to_int.get)(gate_actions)

fig, ax = plt.subplots()
im = ax.imshow(
    gate_Z,
    origin="lower",
    aspect="auto",
    extent=[disagree_grid.min(), disagree_grid.max(), risk_grid.min(), risk_grid.max()],
)

ax.set_title("67.04 — Verifier gate")
ax.set_xlabel("Verifier disagreement")
ax.set_ylabel("Risk score")

cbar = fig.colorbar(im, ax=ax, ticks=list(action_to_int.values()))
cbar.ax.set_yticklabels(list(action_to_int.keys()))

fig.tight_layout()
fig.savefig(FIG_DIR / "67_04_verifier_gate.png", dpi=180)
plt.show()

## 10. Figure 5 — Feature importance

The learned policy should be inspectable. Permutation importance asks how much held-out performance falls when a feature is shuffled.

In [ ]:
importance_result = permutation_importance(
    policy,
    X_test,
    y_test,
    n_repeats=8,
    random_state=67,
    n_jobs=-1,
)

importance = pd.DataFrame({
    "feature": X.columns,
    "importance_mean": importance_result.importances_mean,
    "importance_std": importance_result.importances_std,
}).sort_values("importance_mean", ascending=False)

importance.to_csv(DATA_DIR / "notebook67_policy_feature_importance.csv", index=False)

top = importance.head(12).iloc[::-1]

fig, ax = plt.subplots()
ax.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"])
ax.set_title("67.05 — Policy feature importance")
ax.set_xlabel("Mean held-out accuracy decrease")
ax.set_ylabel("Feature")
fig.tight_layout()
fig.savefig(FIG_DIR / "67_05_feature_importance.png", dpi=180)
plt.show()

importance.head(12)

## 11. Figure 6 — Held-out diagnostics

Notebook 67 uses held-out diagnostics only as a sanity check. Notebook 71 will perform replay, baselines, counterfactual policy scoring, cost-quality curves, and safety/regret analysis.

In [ ]:
report = classification_report(
    y_test,
    pred,
    target_names=label_encoder.classes_,
    output_dict=True,
)

report_df = pd.DataFrame(report).transpose()
report_df.to_csv(DATA_DIR / "notebook67_policy_classification_report.csv")
report_df

In [ ]:
cm = confusion_matrix(y_test, pred)

fig, ax = plt.subplots()
im = ax.imshow(cm)

ax.set_title("67.06 — Learned policy confusion matrix")
ax.set_xticks(range(len(label_encoder.classes_)))
ax.set_yticks(range(len(label_encoder.classes_)))
ax.set_xticklabels(label_encoder.classes_, rotation=35, ha="right")
ax.set_yticklabels(label_encoder.classes_)
ax.set_xlabel("Predicted action")
ax.set_ylabel("Observed / target action")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIG_DIR / "67_06_policy_confusion_matrix.png", dpi=180)
plt.show()

metrics

## 12. Notebook 71 handoff

Notebook 67 exports the policy replay table.

Notebook 71 should evaluate the learned action against baselines:

- static confidence threshold
- entropy threshold
- always continue
- always verify
- expert/hindsight policy
- learned policy

In [ ]:
test_rows = telemetry.loc[test_idx].copy()
test_rows["learned_action"] = label_encoder.inverse_transform(pred)
test_rows["learned_action_probability"] = proba.max(axis=1)

handoff = test_rows[
    [
        "request_id",
        "step",
        "domain",
        "confidence",
        "entropy",
        "margin",
        "risk_score",
        "latency_budget_ms",
        "tokens_so_far",
        "cost_so_far",
        "verifier_disagreement",
        "chosen_action",
        "reward",
        "learned_action",
        "learned_action_probability",
    ]
].copy()

for i, cls in enumerate(label_encoder.classes_):
    handoff[f"p_{cls}"] = proba[:, i]

handoff.to_csv(DATA_DIR / "notebook67_to_71_offline_eval_handoff.csv", index=False)
handoff.head()

In [ ]:
policy_card = {
    "notebook": "67_policy_learning",
    "title": "Policy Learning",
    "repo": "github.com/thinkthoughts/confidence-scheduled-decoding",
    "position": {
        "previous": "61_telemetry_dataset",
        "current": "67_policy_learning",
        "next": "71_offline_evaluation",
    },
    "statement": "Telemetry specifies admissible policy.",
    "features": X.columns.tolist(),
    "actions": label_encoder.classes_.tolist(),
    "metrics": metrics,
    "figures": sorted(p.name for p in FIG_DIR.glob("67_*.png")),
    "handoff_file": "data/notebook67_to_71_offline_eval_handoff.csv",
    "created_at": datetime.utcnow().isoformat() + "Z",
    "notebook_71_questions": [
        "Does the learned policy improve reward at fixed cost?",
        "Does it reduce unnecessary verifier calls?",
        "Does it preserve safety behavior on high-risk rows?",
        "Where does it fail under domain shift?",
        "Which telemetry fields would repair the failures?",
    ],
}

with open(ARTIFACT_DIR / "notebook67_policy_card.json", "w") as f:
    json.dump(policy_card, f, indent=2)

policy_card

## 13. Bridge table

This table keeps the series aligned.

In [ ]:
bridge = pd.DataFrame([
    {
        "from_notebook": 61,
        "artifact": "telemetry rows",
        "to_notebook": 67,
        "role": "learn admissible policy over decoding states",
    },
    {
        "from_notebook": 67,
        "artifact": "policy card + action probabilities + handoff CSV",
        "to_notebook": 71,
        "role": "evaluate learned policy under offline replay",
    },
])

bridge.to_csv(DATA_DIR / "notebook67_bridge_table.csv", index=False)
bridge

## 14. Summary

Notebook 67 defines the policy-learning bridge.

**Result:** telemetry rows become a learned action policy.

**Constraint:** this notebook does not claim deployment readiness.

**Next:** Notebook 71 performs offline evaluation and asks whether the learned policy is admissible under replay.

\[
\text{Telemetry} \rightarrow \hat{\pi}(a \mid s) \rightarrow \text{Offline replay}
\]

## 15. Final Colab download cell

Run this cell at the end of the notebook to package the generated figures, data, artifacts, and notebook.

In [ ]:
zip_name = "notebook67_policy_learning_outputs.zip"
zip_path = ROOT / zip_name

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for folder in [FIG_DIR, DATA_DIR, ARTIFACT_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.relative_to(ROOT))
    notebook_file = ROOT / "67_policy_learning.ipynb"
    if notebook_file.exists():
        z.write(notebook_file, "67_policy_learning.ipynb")

print(f"Wrote {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))